In [1]:
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
from anthropic import Anthropic
import os

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-0"

In [5]:
# helper functions that will help maintain the conversation state with claude
def add_user_message(messages, text):
    user_message = {"role":"user","content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant","content":text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params={
        "model":model,
        "max_tokens":1000,
        "messages":messages,
        "temperature":temperature
    }
    if system:
        params["system"]=system
    if stop_sequences:
        params["stop_sequences"]=stop_sequences
        
    message=client.messages.create(**params)
    
    return message.content[0].text


In [3]:
messages=[]

In [ ]:
for i in range(2):
    print("Enter the question you want to ask")
    user_request = input("Enter the question you want claude to answer: ")
    add_user_message(messages,user_request)
    #without system prompts we have to do this
    assistant_text=chat(messages)
    add_assistant_message(messages,assistant_text)
    print("---")
    print(f"{i+1} text: ",assistant_text)
    print("---")

Enter the question you want to ask
---
1 text:  1 + 1 = 2
---
Enter the question you want to ask
---
2 text:  2 + 2 = 4
---


In [15]:
# UNDERSTANDING SYSTEM PROMPTS IN CLAUDE
system_prompt="""You are a Python Enginner who writes very clean and concise code.
"""
messages.clear()
add_user_message(messages,"Write a python function to add two numbers")
print("messages dictionary:",messages)
#with system prompts we do this
# answer = chat(messages, system=system_prompt)
# print(answer)

messages dictionary: [{'role': 'user', 'content': 'Write a python function to add two numbers'}]


In [9]:
messages.clear()
add_user_message(messages,"write a 1 sentence on moon")
with client.messages.stream(model=model,max_tokens=1000,messages=messages) as stream:
    for text in stream.text_stream:
        print(text, end="")

The moon hung like a luminous pearl in the velvet night sky, casting its gentle silver light across the sleeping world below.

In [6]:
#STRUCTURED DATA
messages.clear()
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages,"```json")
text = chat(messages, stop_sequences=["```"])
print(text)


{
  "Name": "OrderProcessingRule",
  "EventPattern": {
    "source": ["myapp.orders"],
    "detail-type": ["Order Placed"]
  },
  "Targets": [
    {
      "Id": "1",
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"
    }
  ]
}



In [ ]:
#strcutured data exercise
messages.clear()
prompt = """Generate three different AWS CLI commands. Each should be very short"""
add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all the bash commands in a single block without any comments:\n```bash")
# here we add a pre filled message to claude to make generation look good and as we want it
text=chat(messages,stop_sequences=["```","**"])
print(text)


aws s3 ls

aws ec2 describe-instances

aws iam list-users

